# Regresja logistyczna jako pipeline
Ten sam model co w przykladowa regresja logistyczna, ale ladnie ulozony w pipeline

In [5]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2

from funkcje import *
from wskazniki_kredytowe import oblicz_wskazniki

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

## Transformer przygotowania danych



In [7]:

class PrzygotowanieCech(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        kolumny_do_pominiecia=None,
        prog_korelacji=0.9,
        prog_auc=0.6,
        tylko_wskazniki=False,
        prog_nan=0.5,
        prog_zer=0.5,
    ):
        self.kolumny_do_pominiecia = kolumny_do_pominiecia
        self.prog_korelacji = prog_korelacji
        self.prog_auc = prog_auc
        self.tylko_wskazniki = tylko_wskazniki
        self.prog_nan = prog_nan
        self.prog_zer = prog_zer

    def _przygotuj_baze(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Dane wejściowe muszą być obiektem pandas.DataFrame.")

        pomin = self.kolumny_do_pominiecia or []
        brakujace = sorted(set(pomin) - set(X.columns))
        if brakujace:
            raise ValueError(f"Brak wymaganych kolumn do pominięcia: {brakujace}")

        baza = X.drop(columns=pomin).copy()
        kolumny_poczatkowe = set(baza.columns)

        baza = oblicz_wskazniki(baza)

        if self.tylko_wskazniki:
            nowe_kolumny = [
                kolumna for kolumna in baza.columns
                if kolumna not in kolumny_poczatkowe
            ]
            baza = baza[nowe_kolumny]

        return baza

    @staticmethod
    def _kolumny_numeryczne(X):
        kolumny = X.select_dtypes(include=["float"]).columns.tolist()
        if "wsk_liczba_dni_istnienia" in X.columns and "wsk_liczba_dni_istnienia" not in kolumny:
            kolumny.append("wsk_liczba_dni_istnienia")
        return kolumny

    def _odrzuc_kolumny_niskiego_auc(self, X, y):
        rows = []
        for kolumna in X.columns:
            maska = X[kolumna].notna()
            if maska.sum() == 0 or y.loc[maska].nunique() < 2:
                continue
            auc = roc_auc_score(y.loc[maska], X.loc[maska, kolumna])
            rows.append((kolumna, max(auc, 1 - auc), auc >= 0.5))

        if not rows:
            raise ValueError("Nie udało się policzyć AUC dla żadnej cechy.")

        tabela = pd.DataFrame(rows, columns=["cecha", "auc", "zgodny_kierunek"])
        do_usuniecia = tabela.loc[tabela["auc"] < self.prog_auc, "cecha"].tolist()
        do_odwrocenia = tabela.loc[
            (tabela["auc"] >= self.prog_auc) & tabela["zgodny_kierunek"], "cecha"
        ].tolist()
        return do_usuniecia, do_odwrocenia

    def fit(self, X, y):
        y = pd.Series(y, index=X.index if isinstance(X, pd.DataFrame) else None)
        baza = self._przygotuj_baze(X)
        self.kolumny_numeryczne_ = self._kolumny_numeryczne(baza)
        if not self.kolumny_numeryczne_:
            raise ValueError("Nie znaleziono cech numerycznych.")

        numeryczne = baza[self.kolumny_numeryczne_].copy()
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)

        maska_nan = numeryczne.isna().mean() <= self.prog_nan
        maska_zer = numeryczne.eq(0).mean() <= self.prog_zer
        maska_jakosci = maska_nan & maska_zer

        self.kolumny_zbyt_wielu_brakow_ = numeryczne.columns[~maska_jakosci].tolist()
        numeryczne = numeryczne.loc[:, maska_jakosci]

        po_korelacji = usuwanie_najbardziej_scorrelowanych(
            numeryczne, prog=self.prog_korelacji
        )
        self.kolumny_po_korelacji_ = po_korelacji.columns.tolist()
        nie_monotoniczne = wykryj_niemonotoniczne(po_korelacji, y)
        self.kolumny_niemonotoniczne_ = list(nie_monotoniczne)

        if self.kolumny_niemonotoniczne_:
            po_zawiasach, self.progi_ = dodaj_cechy_zawiasowe(
                po_korelacji, y, self.kolumny_niemonotoniczne_
            )
        else:
            po_zawiasach, self.progi_ = po_korelacji, {}

        self.kolumny_do_usuniecia_, self.kolumny_do_odwrocenia_ = (
            self._odrzuc_kolumny_niskiego_auc(po_zawiasach, y)
        )
        wynik = po_zawiasach.drop(columns=self.kolumny_do_usuniecia_)
        wynik[self.kolumny_do_odwrocenia_] = -wynik[self.kolumny_do_odwrocenia_]

        skosnosc = wynik.skew()
        self.kolumny_do_log_ = [
            kolumna for kolumna in wynik.columns
            if skosnosc[kolumna] > 2 and wynik[kolumna].min() >= 0
        ]
        wynik[self.kolumny_do_log_] = np.log1p(wynik[self.kolumny_do_log_])
        self.kolumny_wynikowe_ = wynik.columns.tolist()
        return self

    def transform(self, X):
        baza = self._przygotuj_baze(X)
        numeryczne = baza.reindex(columns=self.kolumny_numeryczne_)
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)
        numeryczne = numeryczne.drop(columns=self.kolumny_zbyt_wielu_brakow_)
        numeryczne = numeryczne.reindex(columns=self.kolumny_po_korelacji_)

        if self.kolumny_niemonotoniczne_:
            numeryczne = dodaj_cechy_zawiasowe(
                numeryczne,
                kolumny_niemonotoniczne=self.kolumny_niemonotoniczne_,
                progi=self.progi_,
            )[0]

        numeryczne = numeryczne.drop(columns=self.kolumny_do_usuniecia_)
        numeryczne[self.kolumny_do_odwrocenia_] = -numeryczne[self.kolumny_do_odwrocenia_]
        numeryczne[self.kolumny_do_log_] = np.log1p(numeryczne[self.kolumny_do_log_])
        return numeryczne.reindex(columns=self.kolumny_wynikowe_)


## Trenowanie i użycie pipeline'u

In [8]:
df = pd.read_csv("zbior_1.csv")
TARGET_COLUMN = df.columns[-1]
KOLUMNY_DO_POMINIECIA = [
    "szczegolnaFormaPrawna_Symbol",
    "schemat_wsk_rzis",
    "schemat_wsk_bilans",
]

X = df.drop(columns=TARGET_COLUMN)
y = df[TARGET_COLUMN]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=2137, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=2137, stratify=y_temp
)

pipeline = Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech(KOLUMNY_DO_POMINIECIA)),
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        l1_ratio=1, solver="liblinear", C=100, random_state=2137
    )),
])

pipeline.fit(X_train, y_train)
prawdopodobienstwo = pipeline.predict_proba(X_valid)[:, 1]
predykcja = pipeline.predict(X_valid)

print(f"AUC: {roc_auc_score(y_valid, prawdopodobienstwo):.3f}")

Usunięto 86 kolumn.
Liczba przeanalizowanych cech: 29
Liczba cech niemonotonicznych: 8


,cecha,liczba_decyl,min_n,DR_min,DR_max,rozstep_DR,wzrosty,spadki,zmiany_kierunku,niemonotoniczna
24,wsk_zadluzenia,10,175,0.0238,0.2381,0.2143,2,2,3,True
4,wsk_mnoznik_kap_wl,10,196,0.0333,0.2238,0.1905,3,1,2,True
8,wsk_zysk_operacyjny_zob_1,10,90,0.0286,0.1857,0.1571,2,2,2,True
17,wsk_plynnosc_szybka_2,10,209,0.0476,0.1571,0.1095,2,1,2,True
19,wsk_udzial_kap_obrotowego_w_fin,10,156,0.0568,0.1657,0.1089,2,1,2,True
26,wsk_fin_majatku_kapitalem,10,187,0.0571,0.1872,0.1300,1,1,1,True
9,wsk_udzial_kap_wlasnego_aktywa_2,10,121,0.0524,0.1818,0.1294,1,1,1,True
0,wsk_index_aktywne,10,203,0.0388,0.1402,0.1014,1,1,1,True


AUC: 0.796


In [ ]:
#optymalizcja parametrow
param_grid = {
    "przygotowanie_cech__prog_auc": [0.55, 0.65],
    "przygotowanie_cech__tylko_wskazniki": [True, False],
    "przygotowanie_cech__prog_nan": [0.3, 0.5],
    "przygotowanie_cech__prog_zer": [0.5, 0.8],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=2,
    refit=True,
)

search.fit(X_train, y_train)

print("Najlepsze parametry:", search.best_params_)
print(f"Najlepszy CV AUC: {search.best_score_:.4f}")

prawdopodobienstwo = search.predict_proba(X_valid)[:, 1]
print(f"Validation AUC: {roc_auc_score(y_valid, prawdopodobienstwo):.4f}")